In [ ]:
import numpy as np

from tqdm import trange
import matplotlib.pyplot as plt

import adaptive_latents
from adaptive_latents import StreamingKalmanFilter, Bubblewrap, ArrayWithTime, Pipeline, proSVD, Tee, VJF, CenteringTransformer
from adaptive_latents.regressions import BaseKNearestNeighborRegressor
from adaptive_latents.stim_regressor import StimRegressor
from tqdm.autonotebook import tqdm

rng = np.random.default_rng(0)


In [ ]:
d = adaptive_latents.datasets.Odoherty21Dataset()

In [ ]:
def S(point, stim):
    return stim


In [ ]:
def do_experiment(input_arrays, decay_rate = .9, stim_scale=1, S=S, rng=None, pairs_to_evaluate_on=None):
    if rng is None:
        rng = np.random.default_rng(0)
    centerer = CenteringTransformer()

    pro = proSVD(k=10)

    sr = StimRegressor(
        autoreg=StreamingKalmanFilter(),
        stim_reg=BaseKNearestNeighborRegressor(k=20, maxlen=1000),
        attempt_correction=True
    )

    stims = []
    predictions = []
    latents = []
    dt_X = []
    s_hat_evals = []
    s_eval = None

    to_add = np.zeros(input_arrays[0].shape[1])

    for input_array in tqdm(input_arrays):
        for data in Pipeline().streaming_run_on(input_array):


            latent_location = pro.transform(centerer.transform(data))

            stim = np.zeros(data.shape[1])
            if rng.random() < .01 and pro.is_initialized:
                stim = pro.Q[:,0] * stim_scale
                to_add += S(latent_location, stim)

            data = data + to_add
            to_add = to_add * decay_rate

            data = centerer.partial_fit_transform(data)
            data = pro.partial_fit_transform(data)
            latents.append(data)

            qX = ArrayWithTime([[1]], data.t)
            sr.partial_fit_transform(np.array([[stim]]), stream='stim')
            prediction = sr.partial_fit_transform(qX, stream='dt_X')
            sr.partial_fit_transform(data, stream='X')


            stims.append(ArrayWithTime(stim, data.t))
            predictions.append(prediction)
            dt_X.append(qX)

            if pairs_to_evaluate_on is not None:
                point_evals = []
                for point, stim in pairs_to_evaluate_on:
                    stim_reg_input = np.hstack([point.flatten(), stim.flatten()])
                    expanded_diff = pro.inverse_transform(sr.stim_reg.predict(stim_reg_input)[None,:])
                    point_evals.append(expanded_diff.flatten())
                s_hat_evals.append(ArrayWithTime(point_evals,data.t))

                if s_eval is None:
                    point_evals = []
                    for point, stim in pairs_to_evaluate_on:
                        point_evals.append(S(point, stim))
                    s_eval = np.array(point_evals)


            if not sr.autoreg.parameter_fitting:
                sr.autoreg.toggle_parameter_fitting(True)
        sr.autoreg.toggle_parameter_fitting(False)


    stims = ArrayWithTime.from_list(stims)
    predictions = ArrayWithTime.from_list(predictions, drop_early_nans=True)
    latents = ArrayWithTime.from_list(latents, squeeze_type='to_2d')
    s_hat_evals = ArrayWithTime.from_list(s_hat_evals, drop_early_nans=True)
    return predictions, latents, stims, (s_hat_evals, s_eval)



In [ ]:
predictions, latents, stims, _ = do_experiment([d.neural_data.slice(slice(0,500))], stim_scale=1)

In [ ]:

edges = np.linspace(d.neural_data.t[0], d.neural_data.t[-1], 6)
i = 2
s = (edges[i] < d.neural_data.t) & (d.neural_data.t < edges[i+1])

stim_slice = stims.any(axis=1)
pairs_to_evaluate_on = list(zip(latents.slice(stim_slice), stims.slice(stim_slice)))

predictions, latents, stims, (s_hat_evals, s_eval) = do_experiment([d.neural_data.slice(s)], pairs_to_evaluate_on=pairs_to_evaluate_on, stim_scale=1)


In [ ]:
s_hat_evals.shape, s_eval.shape

In [ ]:
%matplotlib inline
fig, ax = plt.subplots()
diff = ArrayWithTime.subtract_aligned_indices(latents1, latents2)
ax.plot(diff.t, diff)
ax.set_xlim([75,100])
ax.axhline(-5, color='k')



In [ ]:
%matplotlib inline
fig, ax = plt.subplots()

i = 0
ax.plot(latents.t, latents[:,i])
ax.plot(predictions.t, predictions[:,i])
